# Setup

In [1]:
#!/usr/bin/env python3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent.parent
print("✅ Model training setup complete")   

✅ Model training setup complete


# Load Features

In [2]:
df = pd.read_parquet(PROJECT_ROOT / 'data/features/engineered_features.parquet')
print(f"📊 Loaded {df.shape[0]} samples, {df.shape[1]} features")

# Separate features and target
feature_cols = [col for col in df.columns if col not in ['target']]
X = df[feature_cols].values
y = df['target'].values

# Train/test split
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"📊 Train: {X_train.shape[0]} samples")
print(f"📊 Test: {X_test.shape[0]} samples")


📊 Loaded 3887 samples, 25 features
📊 Train: 3109 samples
📊 Test: 778 samples


# Train Models

In [3]:
models = {
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostRegressor(iterations=100, random_seed=42, verbose=False)
}

results = {}
for name, model in models.items():
    print(f"\n📊 Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'rmse': rmse, 'mae': mae, 'r2': r2, 'model': model}
    print(f"   RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")


📊 Training RandomForest...
   RMSE: 0.0368, MAE: 0.0316, R²: -2.5100

📊 Training XGBoost...
   RMSE: 0.0464, MAE: 0.0402, R²: -4.5851

📊 Training LightGBM...
   RMSE: 0.0336, MAE: 0.0283, R²: -1.9273

📊 Training CatBoost...
   RMSE: 0.0317, MAE: 0.0261, R²: -1.6092


# Model Comparison

In [4]:
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)
comparison = pd.DataFrame(results).T
print(comparison.sort_values('rmse'))


MODEL COMPARISON
                  rmse       mae        r2  \
CatBoost      0.031713  0.026106 -1.609169   
LightGBM      0.033591  0.028315 -1.927343   
RandomForest  0.036783  0.031558 -2.509981   
XGBoost       0.046399  0.040163 -4.585064   

                                                          model  
CatBoost      CatBoostRegressor(iterations=100, loss_functio...  
LightGBM      LGBMRegressor(n_jobs=-1, random_state=42, verb...  
RandomForest  (DecisionTreeRegressor(max_features=1.0, rando...  
XGBoost       XGBRegressor(base_score=None, booster=None, ca...  


# Save Best Model

In [5]:
best_model_name = comparison.sort_values('rmse').index[0]
best_model = results[best_model_name]['model']
print(f"\n🏆 Best Model: {best_model_name}")

# Save model
import joblib
model_path = PROJECT_ROOT / 'models/checkpoints'
model_path.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, model_path / f'{best_model_name.lower()}_model.pkl')
print(f"✅ Model saved to {model_path / f'{best_model_name.lower()}_model.pkl'}")


🏆 Best Model: CatBoost
✅ Model saved to c:\Users\nyvra\Downloads\sp500-predictor\models\checkpoints\catboost_model.pkl
